
# Load FOCUS cost export (flat-folder / manual-upload variant)

This variant is for a **manually downloaded and uploaded** Cost Management export -- one or
more `part_N_NNNN.csv` files sitting directly in a Lakehouse `Files/` folder, with no
date-based subfolder structure. (If your export instead lands automatically via a **OneLake
shortcut** to a scheduled export's storage container -- the recurring, dated-folder layout
FCA's own `01_Load_Focus.Notebook` assumes -- that needs different logic; ask if you want
that mode added here too.)

Same FOCUS schema as before, same `focus_staging` → `focus` Delta table target, so FCA's
existing `01_Load_Focus_Fabric`, semantic model, and report still run on top of it unchanged.

Because there's no per-period folder to detect, this reads and **overwrites the whole `focus`
table** on every run, rather than merging by billing period -- simplest and safest given a
manual, occasional refresh workflow (each time you have newer data, re-upload the file(s) and
re-run this notebook).


## Step 0 – Confirm your exact Fabric filter values before relying on them

In [ ]:
from delta.tables import *
from notebookutils import mssparkutils
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
from pyspark.sql.utils import AnalysisException
from pyspark.sql.functions import col, when, from_json, date_format, lit, row_number, max, lower
from pyspark.sql.types import StructType, StringType
from pyspark.sql.window import Window
import re
import glob

# Folder containing your uploaded CSV file(s) -- a FOLDER, not a specific file. All *.csv
# files directly under it are read (no subfolder/date-pattern detection in this variant).
rawSourcePath = "Files/Data/FCA"

# FOCUS's ServiceName for Fabric has been observed as both "Microsoft Fabric" (FOCUS spec
# convention, no dot) and "Microsoft.Fabric" (Azure resource-provider convention, with a
# dot) depending on export version/tenant -- confirm which one you actually have below
# before relying on the hardcoded filter further down.
fabric_service_names = ["Microsoft Fabric", "Microsoft.Fabric"]

In [ ]:
# One-time check: point this at a single known CSV file from your export (any date folder)
# to confirm the actual values present before the real ingestion loop filters on anything.
sample_csv_path = "<Files/focuscost/.../a-single-export-file>.csv"  # fill in, then run once

sample_df = spark.read.option("header", "true").csv(sample_csv_path)
print("Distinct ServiceName values:")
sample_df.select("ServiceName").distinct().show(truncate=False)

# If your export mixes multiple services (Databricks, Fabric, disk/compute, etc.),
# ServiceName alone may not cleanly isolate Fabric -- ChargeDescription is usually more
# precise for this. Check its exact values (case/spacing matters for an exact-match filter):
print("Distinct ChargeDescription values:")
sample_df.select("ChargeDescription").distinct().show(truncate=False)

Use whichever of the two turns out to isolate Fabric cleanly for your export.

If `ServiceName` mixes Fabric in with other services, filter on `ChargeDescription` instead
in FCA's **`01_Load_Focus_Fabric.Notebook`** (not in this notebook -- this one just loads the
raw `focus` table; the Fabric-only filter lives downstream). Replace its line:

```python
focus_df = focus_df.where(f"""ServiceName = 'Microsoft.Fabric' and BillingPeriodStart IN ({date_condition})""")
```

with the exact values Step 0 showed above, e.g.:

```python
focus_df = focus_df.where(
    f"lower(ChargeDescription) IN ('fabric', 'onelake') and BillingPeriodStart IN ({date_condition})"
)
```

(`lower(...)` guards against case differences like "OneLake" vs "One Lake" vs "One lake" --
safer than an exact-case match once you've confirmed the values, in case they vary slightly
across export runs.) Use the literal values Step 0 printed for you, not the ones in this
example.

In [ ]:
def list_csv_files(path):
    """Every *.csv file directly under path (no recursion into subfolders)."""
    return [entry.path for entry in mssparkutils.fs.ls(path) if entry.isFile and entry.name.endswith(".csv")]

## STEP 1 – Load Silver

Reads every CSV directly under `rawSourcePath` and overwrites the `focus` Delta table.


In [ ]:
csv_files = list_csv_files(rawSourcePath)
print(f"Found {len(csv_files)} CSV file(s) under {rawSourcePath}:")
for f in csv_files:
    print(f"  {f}")

if not csv_files:
    raise ValueError(f"No .csv files found directly under {rawSourcePath} -- check the path and that files were actually uploaded there.")

In [ ]:
df = (
    spark.read
    .option("header", "true")
    .option("multiLine", "true")
    .option("escape", '"')
    .csv(csv_files)
)

df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("focus_staging")

focus_staging_df = DeltaTable.forPath(spark, "Tables/focus_staging").toDF()
focus_staging_df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("focus")

print(f"Loaded {focus_staging_df.count()} rows into 'focus' (full overwrite)")

## Next steps

- Run FCA's existing `01_Load_Focus_Fabric.Notebook` next — it reads from the `focus` table
  this notebook just populated. **Before your first real run**, update its Fabric filter to
  match what Step 0 showed for your export: swap the `ServiceName = 'Microsoft.Fabric'` line
  for a `ChargeDescription`-based filter if your export mixes multiple services -- see Step 0
  above for the exact replacement.
- From there, FCA's semantic model (`FCA_Core_SM`) and report (`FCA_Core_Report`) work as
  documented in [`../../Deploy.md`](../../Deploy.md) — no changes needed.
- **Refresh workflow**: since this is a manual upload rather than a scheduled export, there's
  no automatic "new data" trigger. Each time you want updated numbers, download a fresh CSV
  from Cost Management, upload it into `rawSourcePath` (replacing or alongside the old
  file(s) — this notebook reads everything present and overwrites `focus` each run, so stale
  duplicates from an old file left in that folder would double-count), then re-run this
  notebook and `01_Load_Focus_Fabric`.
- If you'd rather not do that manually each time, setting up the OneLake shortcut to the
  scheduled export's storage container (the automated mode) removes that step — say the word
  if you want that variant built out too.